# Raikou Optical · Phase 0 · Notebook 1 — build the India training set

**Runtime: CPU, High-RAM.** *Runtime → Change runtime type → CPU, then turn on High-RAM.* No GPU: this notebook only downloads and cleans data. A GPU here wastes compute units.

What it does, for each of the 100 tiles (10 × 10 km, native UTM, 10 m):
1. picks the clearest Sentinel-2 L2A pass in each season window of the **2020–21** hydrological year (Oct 15–Nov 30, Jan 15–Feb 28, Apr 1–May 20) from Microsoft Planetary Computer (free, no Copernicus quota), judged by the SCL cloud mask **over the tile itself**;
2. reads the 12 bands in OlmoEarth order plus SCL on the exact tile grid and removes the baseline-04.00 offset where it applies;
3. clips ESA WorldCover 2021 v200 onto the same grid, remaps 11 → 5 classes and builds two label maps: `label_raw` (unfiltered, used for testing and calibration) and `label_clean` (multi-season consensus filter + 1 px boundary ignore, used for training);
4. writes `tiles/<id>.npz` + `.json` and a quicklook PNG to Drive.

Time: about 2–3 h for 100 tiles. **Resumable**: re-run the build cell after a disconnect and finished tiles are skipped. Drive space: about 5 GB.

## 1 · Mount Drive and point Python at the code

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys, json
ROOT = '/content/drive/MyDrive/Raikou_Optical_Phase0'   # change only if you put the folder elsewhere
CODE = f'{ROOT}/optical_phase0'
assert os.path.exists(f'{CODE}/raikou_phase0/__init__.py'), f'Upload the optical_phase0 folder into {ROOT} first (see README).'
os.environ['RAIKOU_P0_ROOT'] = ROOT
if CODE not in sys.path:
    sys.path.insert(0, CODE)

## 2 · Install the geospatial packages (about 1 minute)

In [ ]:
!pip -q install "odc-stac>=0.3.10" "pystac-client>=0.8" "planetary-computer>=1.0" "rasterio>=1.3.10" pyproj folium

## 3 · Self-test (no network, a few seconds)
Checks the tile grid, the WorldCover remap, the offset handling, the label rules on synthetic pixels and the QGIS/ArcGIS style writers. Stop here if anything fails.

In [ ]:
import pandas as pd
from IPython.display import Image, display
from raikou_phase0 import config as C, tiles, build_dataset as B, selftest, gis_export as G
paths = C.Paths().ensure()
selftest.cpu_selftest()

## 4 · The 100-tile manifest
Written once to `manifests/india_100_tiles.csv`. **If you edit that CSV, your copy wins** on every later run. Splits are by whole anchor area (all tiles of one place share a split), and val/test tiles sit at least 20 km from any training tile. Visakhapatnam (coast) and Bellary (Karnataka, Tungabhadra basin) are locked into test.

In [ ]:
if os.path.exists(paths.manifest_csv):
    tile_list = tiles.load_manifest(paths.manifest_csv)
else:
    tile_list = tiles.build_manifest()
    tiles.save_manifest(tile_list, paths.manifest_csv)
report = tiles.check_manifest(tile_list)
print(json.dumps({k: report[k] for k in ('n_tiles', 'per_split', 'per_zone', 'epsg', 'overlaps', 'split_buffer_violations')}, indent=1))
assert report['ok'], 'fix the overlaps / buffer violations listed above'
tiles.folium_map(tile_list)

## 5 · WorldCover preview (a few minutes, before any Sentinel-2 download)
Flags tiles that are mostly water, one class, or mostly no-data/snow. If one looks wrong, move it in the CSV (or in `ANCHORS` in `tiles.py`), delete the CSV, and re-run step 4.

In [ ]:
wc = pd.DataFrame(B.worldcover_preview(tile_list))
wc.to_csv(f'{paths.reports}/worldcover_preview.csv', index=False)
display(wc[wc.flags != ''])
display(wc.groupby('zone')[C.CLASS_NAMES + ['ignored']].mean().round(3))

## 6 · Build all tiles (2–3 h, resumable)
Re-run this cell until it reports 0 failed. A failed tile leaves `tiles/<id>.error.txt` on Drive.

In [ ]:
from tqdm.auto import tqdm
print('Training imagery:', C.TRAIN_HYDRO_YEAR, '->', [(w['name'], w['start'], w['end']) for w in C.season_windows(C.TRAIN_HYDRO_YEAR)])
results = B.build_all(tile_list, paths, workers=3, progress=tqdm)
failed = {k: v for k, v in results.items() if v.startswith('error')}
print(sum(v == 'done' for v in results.values()), 'built now,', sum(v == 'skipped' for v in results.values()), 'already done,', len(failed), 'failed')
failed

## 7 · What the cleaning did
`kept_by_rule_%` is the share of each WorldCover class that the spectral rule kept, per zone. A warning means a rule removes more than 70% of a class: look at the quicklooks, and loosen that class's threshold in `CONSENSUS_RULES` (config.py) if the rule is too strict. Then delete those tiles' `.json` files and rebuild them.

In [ ]:
metas = B.load_all_meta(paths, tile_list)
ret = pd.DataFrame(B.retention_table(metas)); ret.to_csv(f'{paths.reports}/retention_by_zone.csv', index=False)
print('Tiles built:', len(metas), '/', len(tile_list))
display(ret[ret.warn != ''] if (ret.warn != '').any() else ret.head(12))
steps = pd.DataFrame(B.timestep_table(metas)); steps.to_csv(f'{paths.reports}/timesteps.csv', index=False)
display(steps.head(15))
print('missing seasons:', int((steps[['post_monsoon', 'rabi', 'summer']] == 'missing').sum().sum()))

In [ ]:
for tid in ['coast_visakhapatnam_1', 'deccan_bellary_1', 'urban_delhi_ncr_1', 'arid_little_rann_1', 'forest_kodagu_1']:
    p = f'{paths.quicklooks}/{tid}.png'
    if os.path.exists(p):
        display(Image(p))

## 8 · Optional: production parity check (Copernicus vs Planetary Computer)
Production fetches recent imagery from Copernicus (CDSE) with `harmonizeValues: true`. This fetches the **same acquisition** from both sources for three tiles and compares every band pixel by pixel.

* Needs Colab secrets `CDSE_CLIENT_ID` and `CDSE_CLIENT_SECRET` (key icon in the left bar → add both → give this notebook access). Never paste them into a cell.
* Costs about **50 processing units** of the 10,000/month shared with SAR. Set `RUN_PARITY = True` to run.
* Pass = at least 99% of pixels within 1 DN in every band. A consistent offset of about 1000 in one source means harmonization differs; a scattered difference usually means the two archives hold different processing versions of that product. Either way, note it before Phase 1.

In [ ]:
RUN_PARITY = False
PARITY_TILES = ['coast_visakhapatnam_1', 'urban_delhi_ncr_1', 'coast_kochi_1']
PARITY_WINDOW = ('2026-04-01', '2026-05-20')   # recent dates, like production
if RUN_PARITY:
    from google.colab import userdata
    from raikou_phase0 import cdse, s2_pc
    by_id = {t.tile_id: t for t in tile_list}
    print('Estimated cost:', round(sum(cdse.estimate_pu(by_id[t].width, by_id[t].height) for t in PARITY_TILES), 1), 'PU')
    token = cdse.get_token(userdata.get('CDSE_CLIENT_ID'), userdata.get('CDSE_CLIENT_SECRET'))
    cat, out = s2_pc.open_catalog(), {}
    for tid in PARITY_TILES:
        t = by_id[tid]
        acq = s2_pc.pick_acquisition(cat, t, *PARITY_WINDOW)
        pc = s2_pc.load_acquisition(acq, t)
        cd = cdse.fetch(t, token, *cdse.window_around(acq['datetime']))
        rep = cdse.parity(pc['dn'], pc['valid'], cd['dn'], cd['valid'])
        rep.update(date=acq['date'], pc_baseline=pc['baseline'], pc_offset_applied=pc['offset_applied'], cdse_pu_spent=cd['pu_spent'])
        out[tid] = rep
        print(f"{tid}: {'PASS' if rep['pass'] else 'CHECK'}  worst band {rep['worst_band_pct_within_tol']}% within 1 DN  PU {cd['pu_spent']}")
    G.save_json(f'{paths.reports}/parity_pc_vs_cdse.json', out)

Done. Next: **Notebook 2** on an L4 GPU runtime.